# v2 · 05 · The v2 knowledge graph (separate database)
Loads the richer states and mined patterns from `v2_04` into a **separate Neo4j database**, so the v1 graph is never touched.

The v2 graph = the full v1 graph + four new things:
```
(:BehaviourState {code, repayment, usage, payment, p_late_next})      60 shared rich states
(:MonthlyCreditRecord)-[:IN_BEHAVIOUR_STATE]->(:BehaviourState)       180,000 links
(:Motif {pattern, odds_ratio_within_late_groups, ...})                 mined 3-month patterns
(:CreditAccount)-[:SHOWS_MOTIF {start_month}]->(:Motif)                found by walking NEXT_MONTH paths
```
**Before running:** create a second DBMS (e.g. in Neo4j Desktop, port 7688), copy `v2/.env.v2.example` to `v2/.env.v2` and fill it in. The loader refuses to run against the v1 database.

**Setup.**

In [ ]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))      # v2 package (adds v1 src to the path too)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from credit_kg_v2 import load_config, OUT
from credit_kg import config as C, scoring as S, modeling as M
from credit_kg import data as D
from credit_kg_v2 import graph_v2 as GV, rich_states as RS

**Connect to the v2 database** (stops with a clear message if it isn't configured or points at v1).

In [ ]:
drv = GV.get_v2_driver(); run = GV.runner(drv)
run("CALL dbms.components() YIELD name, versions, edition RETURN name, versions, edition")

**Rebuild the training-only inputs** exactly as in `v2_04`: rich-state late probabilities and mined patterns.

In [ ]:
wide, monthly, split = D.load_processed()
w = wide.merge(split); tr = w[w.split == "train"]
late_tr = (tr[[f"status_m{m}" for m in range(1, 7)]] >= 1)
p_late = RS.transition_late_prob(RS.rich_states(tr), late_tr.to_numpy())
mined = RS.mine_motifs(tr, tr[C.TARGET], late_tr.sum(axis=1))
print(len(p_late) - 1, "states with training data |", len(mined), "patterns,", int(mined.adds_beyond_late_count.sum()), "selected")

**Load the v2 graph.** v1's tested loader builds the base graph (idempotent), then the v2 structure is added. Running this twice changes nothing.

In [ ]:
info = GV.load_v2_graph(drv, wide, monthly, split, p_late, mined)
info

**Counts**, including the new node and relationship types.

In [ ]:
counts = {lab: run(f"MATCH (n:{lab}) RETURN count(n) AS n")["n"][0] for lab in ["Customer", "CreditAccount", "MonthlyCreditRecord", "RepaymentState", "BehaviourState", "Motif"]}
counts |= {rel: run(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS n")["n"][0] for rel in ["NEXT_MONTH", "IN_STATE", "IN_BEHAVIOUR_STATE", "SHOWS_MOTIF"]}
pd.Series(counts, name="count").to_frame()

**Parity check 1:** the patterns found by walking `NEXT_MONTH` paths in the graph must equal the pandas patterns, for every customer.

In [ ]:
g = run(GV.read("q_motif_paths.cypher")).dropna()
p = RS.customer_motifs(w)
key = lambda d: set(zip(d.customer_id.astype(int), d.start_month.astype(int), d.motif))
assert key(g) == key(p); print("pattern parity OK:", len(g), "customer-patterns")

**Parity check 2:** population movement between rich states read from the graph (training customers) must equal pandas.

In [ ]:
t = run(GV.read("q_state_transitions.cypher")).set_index("state")
st = RS.rich_states(tr)
pdt = pd.DataFrame({"state": st[:, :-1].ravel(), "late": late_tr.to_numpy()[:, 1:].ravel()}).groupby("state")["late"].agg(["sum", "size"])
assert (t.reindex(pdt.index)[["late_next", "n"]].to_numpy() == pdt.to_numpy()).all(); print("transition parity OK")

**Graph question: which patterns are most common among customers in the riskiest September states?** (pure graph traversal, no labels)

In [ ]:
run('''
MATCH (a:CreditAccount)-[:HAS_MONTHLY_RECORD]->(m:MonthlyCreditRecord {month_index: 6})-[:IN_BEHAVIOUR_STATE]->(s:BehaviourState)
WHERE s.p_late_next > 0.7
MATCH (a)-[:SHOWS_MOTIF]->(p:Motif)
RETURN p.pattern AS pattern, p.direction AS direction, round(p.odds_ratio_within_late_groups, 2) AS odds_ratio, count(DISTINCT a) AS accounts
ORDER BY accounts DESC LIMIT 8''')

**One customer's story from the v2 graph**: six rich states in order plus the patterns they show, with evidence. This is what an explanation can quote.

In [ ]:
cid = int(run("MATCH (a:CreditAccount)-[:SHOWS_MOTIF]->() RETURN a.account_id AS id LIMIT 1")["id"][0])
story = run(GV.read("q_customer_story.cypher"), cid=cid).iloc[0]
print(f"customer {cid}"); [print("  ", m) for m in story.months]
[print("   pattern:", m) for m in story.motifs if m]

**Close.**

In [ ]:
drv.close()